# 23. Merge k Sorted Lists

## Topic Alignment
- **Role Relevance**: Merging multiple sorted streams is critical in external sorting algorithms, distributed systems where data arrives from multiple sorted sources, and log aggregation from multiple servers.
- **Scenario**: Implementing k-way merge in database query optimization, combining sorted results from multiple shards, or merging log files from distributed systems for centralized analysis.

## Metadata Summary
- Source: [LeetCode - Merge k Sorted Lists](https://leetcode.com/problems/merge-k-sorted-lists/)
- Tags: `Linked List`, `Divide and Conquer`, `Heap`, `Priority Queue`, `Merge Sort`
- Difficulty: Hard
- Recommended Priority: High

## Problem Statement
You are given an array of `k` linked-lists `lists`, each linked-list is sorted in ascending order. Merge all the linked-lists into one sorted linked-list and return it.

Input: `lists` - an array of k linked-list heads, each list sorted in ascending order.
Output: The head of the merged sorted linked-list.
Constraints: `k == lists.length`, `0 <= k <= 10^4`, `0 <= lists[i].length <= 500`, `-10^4 <= lists[i][j] <= 10^4`, lists are sorted in ascending order, total number of nodes won't exceed `10^4`.

## Progressive Hints
- Hint 1: The brute force approach of repeatedly merging pairs sequentially is O(k*N) where N is total nodes. Can we do better?
- Hint 2: Use a min-heap (priority queue) to always extract the smallest current element across all k lists. Each extraction/insertion is O(log k).
- Hint 3: Alternative: divide-and-conquer approach by pairing lists and merging recursively, similar to merge sort. This achieves O(N log k).
- Hint 4: For heap approach, push the first node of each non-empty list, then repeatedly extract min, append to result, and push the next node from that list.

## Solution Overview
Two optimal approaches exist: min-heap and divide-and-conquer. The min-heap maintains k elements (heads of k lists) and extracts the minimum in O(log k) time, processing all N nodes for O(N log k) total. Divide-and-conquer pairs lists, merges them (O(n)), and recursively combines pairs, also achieving O(N log k). Both are optimal; heap is more intuitive, divide-and-conquer has better cache locality.

## Detailed Explanation
**Min-Heap Approach:**
1. Create a min-heap (priority queue) and initialize with the first node of each non-empty list. Each entry is (node.val, index, node) to handle comparisons.
2. Use a dummy head for the result list and a current pointer.
3. While heap is not empty:
   - Extract the minimum element (val, idx, node) from heap.
   - Append node to result: `current.next = node`, `current = current.next`.
   - If node.next exists, push (node.next.val, idx, node.next) to heap.
4. Return dummy.next.

**Complexity:** Each of N nodes is pushed and popped once, each operation is O(log k), total O(N log k).

**Divide-and-Conquer Approach:**
1. If lists is empty, return None. If single list, return lists[0].
2. Pair adjacent lists and merge each pair using the standard two-list merge (LC 21).
3. Store merged results in a new array and recursively merge this array.
4. Continue until one list remains.

**Complexity:** Log k levels of merging, each level processes all N nodes, total O(N log k).

**Alternative (Brute Force):** Merge lists sequentially: merge list1 and list2, then merge result with list3, etc. This is O(k*N) because each merge processes increasingly longer lists.

**Key insights:**
- Heap maintains exactly k elements, making extraction very efficient.
- Divide-and-conquer leverages the efficient O(n) two-list merge as a building block.
- Both approaches avoid the quadratic behavior of sequential merging.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Min-Heap | O(N log k) | O(k) | N = total nodes; heap size is k; intuitive. |
| Divide and Conquer | O(N log k) | O(1) or O(log k) | O(1) iterative, O(log k) recursive stack; better cache. |
| Sequential Merge | O(k * N) | O(1) | Merges list1+list2, result+list3, etc.; inefficient. |

## Reference Implementation

In [ ]:
from typing import List, Optional
import heapq


class ListNode:
    """Definition for singly-linked list."""
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def mergeKLists(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    """Merge k sorted lists using min-heap."""
    if not lists:
        return None
    
    # Initialize heap with first node of each list
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    
    # Build result list
    dummy = ListNode(0)
    current = dummy
    
    while heap:
        val, idx, node = heapq.heappop(heap)
        current.next = node
        current = current.next
        
        # Push next node from the same list
        if node.next:
            heapq.heappush(heap, (node.next.val, idx, node.next))
    
    return dummy.next


def mergeKListsDivideConquer(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    """Merge k sorted lists using divide and conquer."""
    if not lists:
        return None
    if len(lists) == 1:
        return lists[0]
    
    # Recursively merge pairs
    while len(lists) > 1:
        merged = []
        for i in range(0, len(lists), 2):
            l1 = lists[i]
            l2 = lists[i + 1] if i + 1 < len(lists) else None
            merged.append(mergeTwoLists(l1, l2))
        lists = merged
    
    return lists[0]


def mergeTwoLists(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    """Merge two sorted linked lists."""
    dummy = ListNode(0)
    current = dummy
    
    while l1 and l2:
        if l1.val < l2.val:
            current.next = l1
            l1 = l1.next
        else:
            current.next = l2
            l2 = l2.next
        current = current.next
    
    current.next = l1 if l1 else l2
    return dummy.next


def mergeKListsSequential(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    """Merge k sorted lists sequentially (less efficient O(k*N))."""
    if not lists:
        return None
    
    result = lists[0]
    for i in range(1, len(lists)):
        result = mergeTwoLists(result, lists[i])
    
    return result

## Validation

In [ ]:
def create_linked_list(values):
    """Helper function to create a linked list from a list of values."""
    if not values:
        return None
    head = ListNode(values[0])
    current = head
    for val in values[1:]:
        current.next = ListNode(val)
        current = current.next
    return head


def linked_list_to_list(head):
    """Helper function to convert linked list to Python list for comparison."""
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result


# Test cases
test_cases = [
    ([[1, 4, 5], [1, 3, 4], [2, 6]], [1, 1, 2, 3, 4, 4, 5, 6]),
    ([], []),
    ([[]], []),
    ([[1], [0]], [0, 1]),
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 2, 3, 4, 5, 6, 7, 8, 9]),
    ([[-2, -1, -1, -1], []], [-2, -1, -1, -1]),
]

for list_values, expected in test_cases:
    # Create lists
    lists = [create_linked_list(vals) for vals in list_values]
    
    # Test heap approach
    lists_copy = [create_linked_list(vals) for vals in list_values]
    result = mergeKLists(lists_copy)
    assert linked_list_to_list(result) == expected, f"Heap failed for {list_values}"
    
    # Test divide and conquer
    lists_copy = [create_linked_list(vals) for vals in list_values]
    result = mergeKListsDivideConquer(lists_copy)
    assert linked_list_to_list(result) == expected, f"D&C failed for {list_values}"
    
    # Test sequential (for small cases)
    if len(list_values) <= 5:  # Skip for large k to save time
        lists_copy = [create_linked_list(vals) for vals in list_values]
        result = mergeKListsSequential(lists_copy)
        assert linked_list_to_list(result) == expected, f"Sequential failed for {list_values}"

print('All tests passed for LC 23.')

## Complexity Analysis
- Time Complexity: O(N log k) for heap and divide-and-conquer where N is total nodes and k is number of lists. Heap: N nodes, each push/pop is O(log k). D&C: log k levels, each processes N nodes.
- Space Complexity: O(k) for heap (storing k elements), O(1) for iterative D&C or O(log k) for recursive D&C (call stack).
- Primary Bottleneck: For k=10^4 small lists, heap initialization dominates. For k=2 very long lists, merge operation dominates. Both approaches handle extremes well.

## Edge Cases & Pitfalls
- Empty lists array should return None.
- Array containing only empty lists should return None.
- Single list should be returned as-is without merging.
- When using heap, must include index in tuple to break ties when values are equal (Python heapq requires all tuple elements comparable).
- In divide-and-conquer, handle odd number of lists by checking bounds when pairing.
- Negative values are allowed; ensure comparisons handle them correctly.

## Follow-up Variants
- Merge k sorted arrays into one sorted array (similar heap approach).
- Find the kth smallest element across k sorted lists without fully merging.
- Merge k sorted streams in an online fashion where lists can grow dynamically.
- Implement external k-way merge for data larger than memory (disk-based).

## Takeaways
- Min-heap is the go-to data structure for efficiently selecting minimum among k elements.
- Divide-and-conquer reduces k-way merge to repeated two-way merges, achieving same asymptotic complexity.
- Sequential merging is deceptively inefficient: O(k*N) vs O(N log k) is significant for large k.
- When using Python's heapq with objects, include a tie-breaker (like index) in the tuple to avoid comparison errors.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 21 | Merge Two Sorted Lists | Base case for D&C approach |
| 378 | Kth Smallest Element in a Sorted Matrix | Min-heap for k-way selection |
| 632 | Smallest Range Covering Elements from K Lists | Min-heap with range tracking |